# 4D interpolation: filling in between time frames

Notebook implementation by Andy Thai

Run from the `notebooks/` folder after `pip install -e .`. Paths are relative to it: datasets live in `../data/`, results go to `../output/`.

The first section runs on the two bundled LV volumes in `../examples/data/` (frames 0 and 15).

## Import libraries

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt

from implicit_heart import ImplicitFunction
from implicit_heart import metrics
from pathlib import Path

## Interpolate between two volumes

In [ ]:
Ma = np.load("../examples/data/M0.npy")
Mb = np.load("../examples/data/M15.npy")
Mab = np.array([Ma, Mb]).astype(np.uint8)  # (T=2, Z, H, W)
print(Mab.shape)

imp = ImplicitFunction(Mab, tmax=1, spacing=(10, 1.5625, 1.5625))  # ACDC voxel size (slice, y, x) in mm
img = imp.interpolate(0.5)  # a full 3D volume halfway between the frames

Path("../output").mkdir(exist_ok=True)
np.save("../output/M0-M15-interp.npy", img)

for i in range(img.shape[0]):
    plt.title(f"slice {i}"); plt.imshow(img[i]); plt.show()

Render the result in 3D from the repo root with `python scripts/render_volume.py output/M0-M15-interp.npy --spacing 10 1.5625 1.5625`.

## Ejection fraction

From the volumes enclosed by the fitted f = 0 surface, from the areas of the contour polygons that become boundary constraints, and from voxel counts of the input masks.

In [ ]:
print("Ejection fraction (implicit surface volumes): %.2f%%" % metrics.ejection_fraction_surface(imp, step=2))
print("Ejection fraction (contour polygon areas): %.2f%%" % metrics.ejection_fraction_polygon(Mab))
print("Mask ejection fraction (voxel counts): %.2f%%" % metrics.ejection_fraction(Mab))

## All time frames

Requires `../data/npy/lv.npy`: binary LV masks shaped `(T, Z, H, W)`.

In [ ]:
volume = np.load("../data/npy/lv.npy")
volume[volume != 0] = 1
print(volume.shape)

imp = ImplicitFunction(volume, tmax=volume.shape[0] - 1)
img = imp.interpolate(0)

for i in range(img.shape[0]):
    plt.title(f"slice {i}"); plt.imshow(img[i]); plt.show()